# 14_optuna_candidate_tuning_260515

Step 14 tunes canonical Step 12c XGBoost candidates with Optuna and compares tuned OOF diagnostics against the fixed-parameter 12c baseline. This is not final model selection, SHAP, segmentation, thresholding, or causal/uplift analysis.

In [1]:
from pathlib import Path
from datetime import datetime
import importlib.util
import json
import math
import subprocess
import time
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display, Markdown

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('ignore', category=UserWarning)

STEP = '14_optuna_candidate_tuning_260515'
RANDOM_STATE = 42
N_TRIALS_PER_SCOPE = 20
TIMEOUT_PER_SCOPE_SECONDS = 900
EXPECTED_ROOT = Path('C:/Code/ott-churn-prediction').resolve()

ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()).resolve()
PARK = (ROOT / 'park.ingyeom').resolve()
NOTEBOOK_PATH = (PARK / 'notebook' / STEP / f'{STEP}.ipynb').resolve()

if ROOT != EXPECTED_ROOT:
    raise SystemExit(f'repo root mismatch: {ROOT}')
if not PARK.exists():
    raise SystemExit(f'park.ingyeom missing: {PARK}')

def inside_park(path):
    path = Path(path).resolve()
    try:
        path.relative_to(PARK)
        return True
    except ValueError:
        return False

def require_inside_park(path):
    path = Path(path).resolve()
    if not inside_park(path):
        raise SystemExit(f'blocked outside park.ingyeom: {path}')
    if '_data' in path.parts or '.tmp' in path.parts:
        raise SystemExit(f'blocked folder read/write: {path}')
    return path

def rel(path):
    return str(Path(path).resolve().relative_to(PARK)).replace('\\', '/')

def read_csv(path):
    return pd.read_csv(require_inside_park(path))

def write_csv(df, path):
    path = require_inside_park(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def write_text(text, path):
    path = require_inside_park(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    return path

def all_status_pass(path):
    path = require_inside_park(path)
    if not path.exists():
        return False
    df = pd.read_csv(path)
    if 'status' in df.columns:
        return bool((df['status'].astype(str).str.upper().str.strip() == 'PASS').all())
    if 'passed' in df.columns:
        return bool(df['passed'].astype(str).str.upper().str.strip().isin(['TRUE', 'PASS', 'YES', '1']).all())
    return False

def choose_output_folder(base):
    base = require_inside_park(base)
    if base.exists() and any(p.is_file() for p in base.iterdir()):
        out = base / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    else:
        out = base
    out.mkdir(parents=True, exist_ok=True)
    return out.resolve()

def detect_latest_valid_12c(base, required):
    base = require_inside_park(base)
    candidates = []
    if base.exists():
        candidates.extend([p for p in base.iterdir() if p.is_dir()])
        candidates.append(base)
    candidates = sorted(candidates, key=lambda p: (p.stat().st_mtime, p.name), reverse=True)
    for folder in candidates:
        if all((folder / f).exists() for f in required):
            return folder.resolve()
    return None

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

OUTPUT_FOLDER = choose_output_folder(PARK / 'reports' / 'models' / STEP)
FIGURE_FOLDER = choose_output_folder(PARK / 'reports' / 'figures' / STEP)
ZIP_BASE = require_inside_park(PARK / 'zip' / f'{STEP}_review_package.zip')
ZIP_BASE.parent.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ZIP_BASE if not ZIP_BASE.exists() else ZIP_BASE.with_name(f'{STEP}_review_package_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip')

paths = {
    'cohort_file': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_conservative_features.csv',
    'cohort_final_checks': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_final_checks.csv',
    '12c_base': PARK / 'reports' / 'models' / '12_model_baseline_comparison_canonical_260514',
    'note_md': PARK / 'note.md',
}
req_12c = [
    '12c_final_checks.csv', '12c_candidate_selection_by_scope.csv', '12c_operating_metrics_at_k.csv',
    '12c_stability_aware_candidate_by_scope.csv', '12c_model_comparison_summary.csv',
    '12c_calibration_decile_summary.csv', '12c_feature_set_by_scope.csv', '12c_cv_split_audit.csv'
]
folder_12c = detect_latest_valid_12c(paths['12c_base'], req_12c)
optuna_available = importlib.util.find_spec('optuna') is not None
xgboost_available = importlib.util.find_spec('xgboost') is not None
source_12c_files_exist = folder_12c is not None and all((folder_12c / f).exists() for f in req_12c)
source_12c_final_checks_pass = bool(folder_12c and all_status_pass(folder_12c / '12c_final_checks.csv'))
stop_reasons = []
if not paths['cohort_file'].exists():
    stop_reasons.append('06 primary main cohort conservative feature file missing')
if not folder_12c:
    stop_reasons.append('12c canonical folder with required files missing')
if folder_12c and not source_12c_final_checks_pass:
    stop_reasons.append('12c_final_checks is not all PASS')
if not optuna_available:
    stop_reasons.append('optuna unavailable')
if not xgboost_available:
    stop_reasons.append('xgboost unavailable')

preflight_rows = [
    {'item': 'repo_root', 'status': 'PASS', 'value': str(ROOT).replace('\\', '/'), 'actual_path': str(ROOT), 'notes': ''},
    {'item': 'output_folder_inside_park', 'status': 'PASS' if inside_park(OUTPUT_FOLDER) else 'FAIL', 'value': rel(OUTPUT_FOLDER), 'actual_path': str(OUTPUT_FOLDER), 'notes': ''},
    {'item': 'figure_folder_inside_park', 'status': 'PASS' if inside_park(FIGURE_FOLDER) else 'FAIL', 'value': rel(FIGURE_FOLDER), 'actual_path': str(FIGURE_FOLDER), 'notes': ''},
    {'item': '06_cohort_file_exists', 'status': 'PASS' if paths['cohort_file'].exists() else 'FAIL', 'value': paths['cohort_file'].exists(), 'actual_path': rel(paths['cohort_file']) if paths['cohort_file'].exists() else '', 'notes': ''},
    {'item': '12c_folder_exists', 'status': 'PASS' if folder_12c else 'FAIL', 'value': bool(folder_12c), 'actual_path': rel(folder_12c) if folder_12c else '', 'notes': ''},
    {'item': '12c_required_files_exist', 'status': 'PASS' if source_12c_files_exist else 'FAIL', 'value': source_12c_files_exist, 'actual_path': rel(folder_12c) if folder_12c else '', 'notes': '; '.join(req_12c)},
    {'item': '12c_final_checks_pass', 'status': 'PASS' if source_12c_final_checks_pass else 'FAIL', 'value': source_12c_final_checks_pass, 'actual_path': rel(folder_12c / '12c_final_checks.csv') if folder_12c else '', 'notes': ''},
    {'item': 'optuna_available', 'status': 'PASS' if optuna_available else 'FAIL', 'value': optuna_available, 'actual_path': '', 'notes': ''},
    {'item': 'xgboost_available', 'status': 'PASS' if xgboost_available else 'FAIL', 'value': xgboost_available, 'actual_path': '', 'notes': ''},
    {'item': 'stop_reason', 'status': 'PASS' if not stop_reasons else 'FAIL', 'value': 'none' if not stop_reasons else ' | '.join(stop_reasons), 'actual_path': '', 'notes': ''},
]
preflight_path = write_csv(pd.DataFrame(preflight_rows), OUTPUT_FOLDER / '14_preflight_input_validation.csv')

if stop_reasons:
    readme = '# 14_optuna_candidate_tuning_260515\n\nStep 14 stopped during preflight.\n\nStop reason: ' + ' | '.join(stop_reasons) + '\n'
    readme_path = write_text(readme, OUTPUT_FOLDER / 'README.md')
    final_path = write_csv(pd.DataFrame([
        {'check_name': 'source_06_file_exists', 'status': 'PASS' if paths['cohort_file'].exists() else 'FAIL', 'value': paths['cohort_file'].exists(), 'notes': ''},
        {'check_name': 'source_12c_files_exist', 'status': 'PASS' if source_12c_files_exist else 'FAIL', 'value': source_12c_files_exist, 'notes': ''},
        {'check_name': 'source_12c_final_checks_pass', 'status': 'PASS' if source_12c_final_checks_pass else 'FAIL', 'value': source_12c_final_checks_pass, 'notes': ''},
        {'check_name': 'optuna_available', 'status': 'PASS' if optuna_available else 'FAIL', 'value': optuna_available, 'notes': ''},
        {'check_name': 'xgboost_available', 'status': 'PASS' if xgboost_available else 'FAIL', 'value': xgboost_available, 'notes': ''},
    ]), OUTPUT_FOLDER / '14_final_checks.csv')
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [NOTEBOOK_PATH, preflight_path, readme_path, final_path]:
            if Path(p).exists():
                zf.write(p, rel(p))
    raise SystemExit('Step 14 stopped: ' + ' | '.join(stop_reasons))

import optuna
from xgboost import XGBClassifier

cohort = read_csv(paths['cohort_file'])
cand12 = read_csv(folder_12c / '12c_candidate_selection_by_scope.csv')
ops12 = read_csv(folder_12c / '12c_operating_metrics_at_k.csv')
model12 = read_csv(folder_12c / '12c_model_comparison_summary.csv')
feature12 = read_csv(folder_12c / '12c_feature_set_by_scope.csv')
cv12 = read_csv(folder_12c / '12c_cv_split_audit.csv')

n_splits = int(cv12['fold_id'].max()) if 'fold_id' in cv12.columns and len(cv12) else 5
n_splits_note = f'using 12c fold_id max={n_splits}' if n_splits else '12c n_splits unavailable, using 5'
if not n_splits:
    n_splits = 5

scope_order = ['overall_without_promotion', 'overall_with_promotion', 'promotion_only', 'nonpromotion_only']
feature_map = {r['dataset_scope']: [x for x in str(r['features']).split(';') if x] for _, r in feature12.iterrows()}

for required_col in ['USER_KEY', 'is_repurchase', 'is_promotion']:
    if required_col not in cohort.columns:
        raise SystemExit(f'required column missing from cohort file: {required_col}')

def build_scope_dataset(scope):
    df = cohort.copy()
    if scope == 'promotion_only':
        df = df[df['is_promotion'] == 1].copy()
    elif scope == 'nonpromotion_only':
        df = df[df['is_promotion'] == 0].copy()
    features = feature_map[scope]
    missing = [c for c in features + ['is_repurchase', 'USER_KEY'] if c not in df.columns]
    if missing:
        raise SystemExit(f'missing columns for {scope}: {missing}')
    X = df[features].copy().fillna(0)
    y = df['is_repurchase'].astype(int).to_numpy()
    groups = df['USER_KEY'].astype(str).to_numpy()
    return df.reset_index(drop=True), X.reset_index(drop=True), y, groups, features

scope_summary_rows = []
plan_rows = []
datasets = {}
for scope in scope_order:
    df, X, y, groups, features = build_scope_dataset(scope)
    duplicated_extra = int(df['USER_KEY'].duplicated().sum()) if 'USER_KEY' in df.columns else ''
    scope_summary_rows.append({
        'dataset_scope': scope,
        'row_count': len(df),
        'feature_count': len(features),
        'target_positive_count': int(y.sum()),
        'target_positive_rate': float(y.mean()),
        'target_negative_count': int((1 - y).sum()),
        'target_negative_rate': float((1 - y).mean()),
        'unique USER_KEY count': int(pd.Series(groups).nunique()),
        'duplicated USER_KEY extra rows if calculable': duplicated_extra,
        'is_promotion feature included yes/no': 'yes' if 'is_promotion' in features else 'no',
    })
    c = cand12[cand12['dataset_scope'] == scope].iloc[0]
    selected = c['highest_auc_candidate'] if c['highest_auc_candidate'] == c['operating_metric_candidate'] else c['recommended_candidate_for_14']
    tune_status = 'planned' if selected == 'XGBoost' else 'skipped_non_xgboost_candidate'
    plan_rows.append({
        'dataset_scope': scope,
        '12c highest_auc_candidate': c['highest_auc_candidate'],
        '12c operating_metric_candidate': c['operating_metric_candidate'],
        'selected_tuning_model': selected,
        'reason': '12c highest AUC and operating metric candidate are XGBoost' if selected == 'XGBoost' else '12c candidate was not XGBoost; Step 14 does not expand additional model families',
        'tune_status': tune_status,
    })
    datasets[scope] = (df, X, y, groups, features, selected)

scope_summary_path = write_csv(pd.DataFrame(scope_summary_rows), OUTPUT_FOLDER / '14_scope_dataset_summary.csv')
plan_path = write_csv(pd.DataFrame(plan_rows), OUTPUT_FOLDER / '14_tuning_candidate_plan.csv')

def make_model(params):
    base = dict(
        objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE,
        n_jobs=2, tree_method='hist', verbosity=0
    )
    base.update(params)
    return XGBClassifier(**base)

def metric_pack(y_true, score):
    score = np.clip(np.asarray(score), 1e-7, 1 - 1e-7)
    return {
        'auc': float(roc_auc_score(y_true, score)),
        'ap': float(average_precision_score(y_true, score)),
        'brier': float(brier_score_loss(y_true, score)),
        'logloss': float(log_loss(y_true, score, labels=[0, 1])),
    }

def suggest_params(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
        'subsample': trial.suggest_float('subsample', 0.60, 1.00),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.60, 1.00),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True),
    }

trial_rows = []
best_rows = []
fold_rows = []
summary_rows = []
oof_frames = []
skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

for scope in scope_order:
    df, X, y, groups, features, selected = datasets[scope]
    if selected != 'XGBoost':
        continue

    def objective(trial):
        params = suggest_params(trial)
        valid_aucs = []
        train_aucs = []
        aps = []
        briers = []
        for train_idx, valid_idx in skf.split(X, y, groups):
            model = make_model(params)
            model.fit(X.iloc[train_idx], y[train_idx])
            valid_score = model.predict_proba(X.iloc[valid_idx])[:, 1]
            train_score = model.predict_proba(X.iloc[train_idx])[:, 1]
            vm = metric_pack(y[valid_idx], valid_score)
            tm = metric_pack(y[train_idx], train_score)
            valid_aucs.append(vm['auc'])
            train_aucs.append(tm['auc'])
            aps.append(vm['ap'])
            briers.append(vm['brier'])
        trial.set_user_attr('mean_ap', float(np.mean(aps)))
        trial.set_user_attr('mean_brier', float(np.mean(briers)))
        trial.set_user_attr('mean_train_auc', float(np.mean(train_aucs)))
        trial.set_user_attr('mean_valid_auc', float(np.mean(valid_aucs)))
        trial.set_user_attr('train_valid_gap', float(np.mean(train_aucs) - np.mean(valid_aucs)))
        trial.set_user_attr('fold_auc_std', float(np.std(valid_aucs, ddof=0)))
        return float(np.mean(valid_aucs))

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    start = time.time()
    study.optimize(objective, n_trials=N_TRIALS_PER_SCOPE, timeout=TIMEOUT_PER_SCOPE_SECONDS, show_progress_bar=False)
    elapsed = time.time() - start
    for t in study.trials:
        trial_rows.append({
            'dataset_scope': scope,
            'trial_number': t.number,
            'trial_value_auc': t.value,
            'params': json.dumps(t.params, ensure_ascii=False, sort_keys=True),
            'trial_status': str(t.state),
            'elapsed_time if available': elapsed,
        })
    best_params = dict(study.best_trial.params)
    completed = [t for t in study.trials if str(t.state).endswith('COMPLETE')]
    best_rows.append({
        'dataset_scope': scope,
        'selected_model': 'XGBoost',
        'best_trial': study.best_trial.number,
        'best_params': json.dumps(best_params, ensure_ascii=False, sort_keys=True),
        'best_cv_auc': float(study.best_value),
        'n_trials_completed': len(completed),
    })

    oof = np.full(len(df), np.nan)
    fold_assign = np.full(len(df), -1)
    fold_valid_aucs = []
    fold_train_aucs = []
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y, groups), start=1):
        model = make_model(best_params)
        model.fit(X.iloc[train_idx], y[train_idx])
        valid_score = model.predict_proba(X.iloc[valid_idx])[:, 1]
        train_score = model.predict_proba(X.iloc[train_idx])[:, 1]
        vm = metric_pack(y[valid_idx], valid_score)
        tm = metric_pack(y[train_idx], train_score)
        oof[valid_idx] = valid_score
        fold_assign[valid_idx] = fold
        fold_valid_aucs.append(vm['auc'])
        fold_train_aucs.append(tm['auc'])
        fold_rows.append({
            'dataset_scope': scope, 'fold': fold, 'model_name': 'XGBoost_tuned',
            'auc_valid': vm['auc'], 'ap_valid': vm['ap'], 'brier_valid': vm['brier'], 'logloss_valid': vm['logloss'],
            'auc_train': tm['auc'], 'ap_train': tm['ap'], 'brier_train': tm['brier'], 'logloss_train': tm['logloss'],
            'train_valid_auc_gap': tm['auc'] - vm['auc'],
        })
    om = metric_pack(y, oof)
    summary_rows.append({
        'dataset_scope': scope, 'model_name': 'XGBoost_tuned',
        'oof_auc': om['auc'], 'oof_ap': om['ap'], 'oof_brier': om['brier'], 'oof_logloss': om['logloss'],
        'mean_train_auc': float(np.mean(fold_train_aucs)), 'mean_valid_auc': float(np.mean(fold_valid_aucs)),
        'train_valid_auc_gap': float(np.mean(fold_train_aucs) - np.mean(fold_valid_aucs)),
        'fold_auc_std': float(np.std(fold_valid_aucs, ddof=0)),
        'row_count': len(df), 'feature_count': len(features),
    })
    id_cols = [c for c in ['source_row_number', 'USER_KEY'] if c in df.columns]
    odf = df[id_cols + ['is_repurchase']].copy()
    odf.insert(0, 'dataset_scope', scope)
    odf['repurchase_score'] = oof
    odf['churn_risk'] = 1 - odf['repurchase_score']
    odf['fold'] = fold_assign
    odf['model_name'] = 'XGBoost_tuned'
    oof_frames.append(odf)

trial_path = write_csv(pd.DataFrame(trial_rows), OUTPUT_FOLDER / '14_optuna_trial_summary.csv')
best_params_path = write_csv(pd.DataFrame(best_rows), OUTPUT_FOLDER / '14_best_params_by_scope.csv')
fold_metrics_path = write_csv(pd.DataFrame(fold_rows), OUTPUT_FOLDER / '14_fold_metrics.csv')
model_summary = pd.DataFrame(summary_rows)
model_summary_path = write_csv(model_summary, OUTPUT_FOLDER / '14_model_summary_by_scope.csv')
oof_all = pd.concat(oof_frames, ignore_index=True)
oof_path = write_csv(oof_all, OUTPUT_FOLDER / '14_oof_predictions.csv')

def operating_metrics(df_scope, score_col='churn_risk'):
    rows = []
    total_neg = int((1 - df_scope['is_repurchase']).sum())
    base_neg_rate = total_neg / len(df_scope)
    ordered = df_scope.sort_values(score_col, ascending=False).reset_index(drop=True)
    for frac, label in [(0.05, 'top5'), (0.10, 'top10'), (0.20, 'top20')]:
        n = max(1, int(round(len(ordered) * frac)))
        top = ordered.head(n)
        neg = int((1 - top['is_repurchase']).sum())
        precision = neg / n
        recall = neg / total_neg if total_neg else np.nan
        lift = precision / base_neg_rate if base_neg_rate else np.nan
        rows.append({
            'model_name': 'XGBoost_tuned', 'k_label': label, 'selected_n': n,
            'nonrepurchase_events': neg, 'precision_at_k': precision, 'recall_at_k': recall,
            'lift_at_k': lift, 'base_nonrepurchase_rate': base_neg_rate,
            'mean_churn_risk': float(top['churn_risk'].mean()), 'mean_repurchase_score': float(top['repurchase_score'].mean()),
        })
    return rows

op_rows = []
cal_rows = []
for scope, part in oof_all.groupby('dataset_scope'):
    for r in operating_metrics(part):
        r['dataset_scope'] = scope
        op_rows.append(r)
    for score_type, asc in [('repurchase_score', True), ('churn_risk', True)]:
        ranked = part.sort_values(score_type, ascending=asc).reset_index(drop=True)
        ranked['decile'] = pd.qcut(np.arange(len(ranked)), 10, labels=False) + 1
        if score_type == 'repurchase_score':
            ranked['decile'] = 11 - ranked['decile']
        for decile, d in ranked.groupby('decile'):
            rep_rate = float(d['is_repurchase'].mean())
            cal_rows.append({
                'dataset_scope': scope, 'score_type': score_type, 'decile': int(decile), 'row_count': len(d),
                'observed_repurchase_rate': rep_rate, 'observed_nonrepurchase_rate': 1 - rep_rate,
                'mean_repurchase_score': float(d['repurchase_score'].mean()), 'mean_churn_risk': float(d['churn_risk'].mean()),
            })
operating = pd.DataFrame(op_rows)[['dataset_scope', 'model_name', 'k_label', 'selected_n', 'nonrepurchase_events', 'precision_at_k', 'recall_at_k', 'lift_at_k', 'base_nonrepurchase_rate', 'mean_churn_risk', 'mean_repurchase_score']]
operating_path = write_csv(operating, OUTPUT_FOLDER / '14_operating_metrics_at_k.csv')
calibration_path = write_csv(pd.DataFrame(cal_rows), OUTPUT_FOLDER / '14_calibration_decile_summary.csv')

compare_rows = []
op_compare_rows = []
for _, r14 in model_summary.iterrows():
    scope = r14['dataset_scope']
    candidate = cand12.loc[cand12['dataset_scope'] == scope, 'highest_auc_candidate'].iloc[0]
    r12 = model12[(model12['dataset_scope'] == scope) & (model12['model_name'] == candidate)].iloc[0]
    delta_auc = r14['oof_auc'] - safe_float(r12['oof_auc'])
    delta_ap = r14['oof_ap'] - safe_float(r12['oof_average_precision'])
    delta_brier = r14['oof_brier'] - safe_float(r12['oof_brier_score_loss'])
    gap_change = r14['train_valid_auc_gap'] - safe_float(r12['mean_train_valid_gap'])
    if delta_auc >= 0.005 and gap_change <= 0.01:
        interp = 'meaningful_auc_gain_without_large_gap_increase'
    elif delta_auc > 0 and gap_change <= 0.02:
        interp = 'small_auc_gain_limited_practical_improvement'
    elif delta_auc > 0 and gap_change > 0.02:
        interp = 'auc_gain_with_stability_caution'
    else:
        interp = 'no_auc_improvement_over_12c_fixed_baseline'
    compare_rows.append({
        'dataset_scope': scope, '12c_candidate_model': candidate, '12c_oof_auc': safe_float(r12['oof_auc']),
        '14_tuned_model': r14['model_name'], '14_oof_auc': r14['oof_auc'], 'delta_auc': delta_auc,
        '12c_ap': safe_float(r12['oof_average_precision']), '14_ap': r14['oof_ap'], 'delta_ap': delta_ap,
        '12c_brier': safe_float(r12['oof_brier_score_loss']), '14_brier': r14['oof_brier'], 'delta_brier': delta_brier,
        '12c_train_valid_gap': safe_float(r12['mean_train_valid_gap']), '14_train_valid_gap': r14['train_valid_auc_gap'],
        'gap_change': gap_change,
        '12c_fold_auc_std': safe_float(r12['std_valid_auc']), '14_fold_auc_std': r14['fold_auc_std'],
        'fold_auc_std_change': r14['fold_auc_std'] - safe_float(r12['std_valid_auc']),
        'interpretation': interp,
    })
    for label, frac in [('top5', 0.05), ('top10', 0.10), ('top20', 0.20)]:
        r14op = operating[(operating['dataset_scope'] == scope) & (operating['k_label'] == label)].iloc[0]
        r12op = ops12[(ops12['dataset_scope'] == scope) & (ops12['model_name'] == candidate) & (np.isclose(ops12['k_fraction'].astype(float), frac))]
        if r12op.empty:
            lift12 = np.nan; prec12 = np.nan
        else:
            lift12 = safe_float(r12op.iloc[0]['lift_at_k']); prec12 = safe_float(r12op.iloc[0]['precision_at_k'])
        op_compare_rows.append({
            'dataset_scope': scope, 'k_label': label,
            '12c_lift_at_k': lift12, '14_lift_at_k': r14op['lift_at_k'], 'delta_lift_at_k': r14op['lift_at_k'] - lift12 if not np.isnan(lift12) else np.nan,
            '12c_precision_at_k': prec12, '14_precision_at_k': r14op['precision_at_k'], 'delta_precision_at_k': r14op['precision_at_k'] - prec12 if not np.isnan(prec12) else np.nan,
        })
comparison = pd.DataFrame(compare_rows)
comparison_path = write_csv(comparison, OUTPUT_FOLDER / '14_vs_12c_comparison.csv')
op_compare_path = write_csv(pd.DataFrame(op_compare_rows), OUTPUT_FOLDER / '14_vs_12c_operating_comparison.csv')

rec_rows = []
for _, c in comparison.iterrows():
    use16 = 'yes' if c['delta_auc'] >= 0.002 and c['gap_change'] <= 0.02 else 'no'
    if use16 == 'yes':
        rec = 'consider_tuned_candidate_for_16_SHAP_after_review'
        reason = 'Tuned candidate improved AUC enough for follow-up review without excessive train-valid gap increase.'
    else:
        rec = 'prefer_12c_fixed_or_review_before_SHAP'
        reason = 'Tuning did not materially improve AUC or introduced stability caution.'
    rec_rows.append({
        'dataset_scope': c['dataset_scope'], '12c_candidate': c['12c_candidate_model'], '14_tuned_candidate': c['14_tuned_model'],
        'recommendation': rec, 'reason': reason,
        'use_for_16_SHAP_candidate yes/no': use16, 'use_for_final_model no': 'no',
        'caution': 'Step 14 is sensitivity/tuning evidence only; not final model, threshold, segment, or causal evidence.',
    })
recommendation_path = write_csv(pd.DataFrame(rec_rows), OUTPUT_FOLDER / '14_candidate_recommendation_summary.csv')

safe_rows = [
    {'safe wording': 'Step 14는 12c 후보 모델의 Optuna 튜닝 민감도를 확인한 단계다.', 'unsafe wording': 'Optuna로 최종 모델을 확정했다.', 'reason': 'final model selection is not part of Step 14'},
    {'safe wording': '튜닝 결과는 12c fixed-parameter baseline 대비 성능과 안정성 변화를 비교하기 위한 것이다.', 'unsafe wording': 'XGBoost가 최종 모델이다.', 'reason': 'candidate only'},
    {'safe wording': 'top-k churn_risk 지표는 운영 진단용이며, 최종 캠페인 threshold가 아니다.', 'unsafe wording': 'top10 churn_risk를 캠페인 대상으로 확정했다.', 'reason': 'top-k is diagnostic only'},
    {'safe wording': 'AUC 개선이 있더라도 train-valid gap과 fold stability를 함께 봐야 한다.', 'unsafe wording': 'AUC가 올랐으니 개선이다.', 'reason': 'stability matters'},
    {'safe wording': '최종 모델, SHAP 해석, 세그먼트 정책은 후속 단계에서 별도로 확정해야 한다.', 'unsafe wording': 'SHAP 결과에 따르면 원인이 밝혀졌다.', 'reason': 'SHAP was not performed'},
    {'safe wording': '분석 단위는 row-level / subscription-event-level / membership-event-level이다.', 'unsafe wording': 'unique user 기준이다.', 'reason': 'rows are not unique users'},
]
safe_path = write_csv(pd.DataFrame(safe_rows), OUTPUT_FOLDER / '14_safe_unsafe_wording.csv')

risk_rows = [
    {'risk': 'small AUC gains may not be material', 'why_it_matters': 'Tiny gains can be noise relative to fold stability.', 'next_step': 'Review 14_vs_12c_comparison before SHAP.'},
    {'risk': 'train-valid gap can increase after tuning', 'why_it_matters': 'Higher gap weakens generalization confidence.', 'next_step': 'Prefer candidates with bounded gap and stable folds.'},
    {'risk': 'top-k is not a campaign threshold', 'why_it_matters': 'Diagnostic lift is not deployment policy.', 'next_step': 'Define threshold only in a separate operating-policy step.'},
    {'risk': 'calibration is descriptive only', 'why_it_matters': 'Deciles do not guarantee calibrated deployment probabilities.', 'next_step': 'Calibrate only if needed after final candidate review.'},
    {'risk': 'SHAP not performed', 'why_it_matters': 'No feature attribution claim is available yet.', 'next_step': 'Run Step 16 only after candidate confirmation.'},
    {'risk': 'segmentation not created', 'why_it_matters': 'No segment assignment exists from Step 14.', 'next_step': 'Keep segmentation for Step 17 later.'},
    {'risk': 'review/content caveats remain', 'why_it_matters': 'No review columns or new content features were added.', 'next_step': 'Handle content/review in separate audited scope only.'},
]
risk_path = write_csv(pd.DataFrame(risk_rows), OUTPUT_FOLDER / '14_open_risks_for_next_steps.csv')

viz_warn_rows = []
available_fonts = {f.name for f in fm.fontManager.ttflist}
font_name = 'Malgun Gothic' if 'Malgun Gothic' in available_fonts else 'DejaVu Sans'
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False
viz_warn_rows.append({'warning_type': 'font', 'message': f'Using {font_name}'})
figure_rows = []
def save_fig(name, title, source):
    path = FIGURE_FOLDER / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.close()
    figure_rows.append({'figure_file': name, 'relative_path': rel(path), 'title': title, 'source': source})
    return path
try:
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(comparison['dataset_scope'], comparison['delta_auc'], color='#378ADD')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('14 tuned AUC delta vs 12c')
    ax.set_ylabel('delta AUC')
    ax.tick_params(axis='x', rotation=20)
    save_fig('14_fig_01_auc_delta_vs_12c.png', '14 tuned AUC delta vs 12c', '14_vs_12c_comparison.csv')
    top10 = pd.read_csv(op_compare_path)
    top10 = top10[top10['k_label'] == 'top10']
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(top10['dataset_scope'], top10['delta_lift_at_k'], color='#1D9E75')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title('top10 lift delta vs 12c')
    ax.set_ylabel('delta lift@10')
    ax.tick_params(axis='x', rotation=20)
    save_fig('14_fig_02_lift10_delta_vs_12c.png', 'top10 lift delta vs 12c', '14_vs_12c_operating_comparison.csv')
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.bar(model_summary['dataset_scope'], model_summary['train_valid_auc_gap'], color='#D4537E')
    ax.set_title('14 tuned train-valid AUC gap')
    ax.set_ylabel('train-valid gap')
    ax.tick_params(axis='x', rotation=20)
    save_fig('14_fig_03_train_valid_gap_by_scope.png', '14 tuned train-valid AUC gap', '14_model_summary_by_scope.csv')
    rec = pd.read_csv(recommendation_path)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.axis('off')
    lines = [f"{r['dataset_scope']}: 12c={r['12c_candidate']} | 14={r['14_tuned_candidate']} | {r['recommendation']}" for _, r in rec.iterrows()]
    ax.text(0.01, 0.95, '\n'.join(lines), va='top', ha='left', fontsize=10)
    ax.set_title('candidate recommendation summary')
    save_fig('14_fig_04_candidate_recommendation.png', 'candidate recommendation summary', '14_candidate_recommendation_summary.csv')
except Exception as exc:
    viz_warn_rows.append({'warning_type': 'figure_generation', 'message': str(exc)})
figure_inventory_path = write_csv(pd.DataFrame(figure_rows), OUTPUT_FOLDER / '14_figure_inventory.csv')
viz_warn_path = write_csv(pd.DataFrame(viz_warn_rows), OUTPUT_FOLDER / '14_visualization_warnings.csv')

main_result_lines = []
for _, r in comparison.iterrows():
    main_result_lines.append(f"- {r['dataset_scope']}: 12c AUC {r['12c_oof_auc']:.6f} -> 14 AUC {r['14_oof_auc']:.6f}, delta {r['delta_auc']:.6f}, gap change {r['gap_change']:.6f}, interpretation `{r['interpretation']}`")

readme = f"""# {STEP}

## purpose
Step 14는 12c 후보 모델의 Optuna 튜닝 민감도를 확인한 단계다. 튜닝 결과는 12c fixed-parameter baseline 대비 성능과 안정성 변화를 비교하기 위한 것이다.

## input files with actual paths
- 06 cohort file: `{rel(paths['cohort_file'])}`
- 12c folder: `{rel(folder_12c)}`
- 12c candidate selection: `{rel(folder_12c / '12c_candidate_selection_by_scope.csv')}`
- 12c model comparison: `{rel(folder_12c / '12c_model_comparison_summary.csv')}`
- 12c operating metrics: `{rel(folder_12c / '12c_operating_metrics_at_k.csv')}`

## canonical/deprecated 기준
05b, 06, 09b, 11b, 11b semantic patch, 12c canonical outputs만 기준으로 사용했다. old Step 11, old Step 12, old Step 12r, preliminary full-feature model metric은 final evidence로 사용하지 않았다.

## tuning scope
Scopes: {', '.join(scope_order)}. 분석 단위는 row-level / subscription-event-level / membership-event-level이다.

## model candidates
12c candidate file에서 4개 scope 모두 highest AUC candidate 및 operating metric candidate가 XGBoost로 확인되어 XGBoost만 튜닝했다.

## Optuna search space
n_estimators 100-800, max_depth 2-6, learning_rate 0.01-0.20 log, subsample 0.60-1.00, colsample_bytree 0.60-1.00, min_child_weight 1-20, gamma 0-5, reg_alpha 1e-8-10 log, reg_lambda 1e-8-20 log. Actual trials per scope: {N_TRIALS_PER_SCOPE}. Timeout per scope: {TIMEOUT_PER_SCOPE_SECONDS} seconds.

## CV policy
StratifiedGroupKFold, group key USER_KEY, n_splits={n_splits}, random_state={RANDOM_STATE}. {n_splits_note}. USER_KEY는 feature가 아니라 group-aware CV용 key로만 사용했다.

## main results
{chr(10).join(main_result_lines)}

## 12c 대비 개선 여부
AUC 개선이 있더라도 train-valid gap과 fold stability를 함께 봐야 한다. `14_vs_12c_comparison.csv`와 `14_candidate_recommendation_summary.csv`를 기준으로 후속 SHAP 후보 여부를 판단한다.

## top-k diagnostic 해석
top-k churn_risk 지표는 운영 진단용이며, 최종 캠페인 threshold가 아니다. segment assignment도 아니다.

## calibration caveat
decile summary는 descriptive diagnostic이며 deployment calibration guarantee가 아니다.

## interpretation limits
최종 모델, SHAP 해석, 세그먼트 정책은 후속 단계에서 별도로 확정해야 한다. Step 14는 SHAP, segmentation, campaign threshold, causal/uplift 분석을 수행하지 않았다.

## next recommended step
14 결과를 검토한 뒤 `use_for_16_SHAP_candidate yes/no`가 yes인 scope만 Step 16 SHAP 후보로 고려한다. 최종 모델 또는 운영 threshold는 별도 단계에서 결정한다.
"""
readme_path = write_text(readme, OUTPUT_FOLDER / 'README.md')

features_all = set()
for fs in feature_map.values():
    features_all.update(fs)
review_terms = ['review', 'rating', 'comment', 'reply']
time_forbidden_terms = ['response', 'w4', 'week4', '4th', 'day21', 'day_21']
review_cols_used = [f for f in features_all if any(t in f.lower() for t in review_terms)]
forbidden_cols_used = [f for f in features_all if any(t in f.lower() for t in review_terms + time_forbidden_terms)]
time_forbidden_cols_used = [f for f in features_all if any(t in f.lower() for t in time_forbidden_terms)]
groupwise_ok = all(('is_promotion' not in feature_map[s]) for s in ['promotion_only', 'nonpromotion_only'])
churn_check = bool(np.allclose(oof_all['churn_risk'], 1 - oof_all['repurchase_score']))
fold_based = bool(oof_all['fold'].ge(1).all() and not oof_all['repurchase_score'].isna().any())
no_py_created = not any((PARK / 'notebook' / STEP).glob('*.py')) and not any(OUTPUT_FOLDER.glob('*.py'))
all_outputs = [OUTPUT_FOLDER, FIGURE_FOLDER, NOTEBOOK_PATH, ZIP_PATH]
all_inside = all(inside_park(p) for p in all_outputs)

note_section = f"""

## {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {STEP}

### purpose
12c canonical 후보 모델인 XGBoost에 대해 Optuna 튜닝 민감도를 확인하고, 12c fixed-parameter baseline 대비 성능과 안정성 변화를 비교했습니다.

### input canonical files
- 06 cohort: `{rel(paths['cohort_file'])}`
- 12c canonical folder: `{rel(folder_12c)}`
- 12c candidate selection/model comparison/operating/calibration files used from the canonical folder above.

### actual output folder
`{rel(OUTPUT_FOLDER)}`

### actual notebook path
`{rel(NOTEBOOK_PATH)}`

### model/scopes tuned
XGBoost tuned for {', '.join(scope_order)}.

### Optuna trial count
{N_TRIALS_PER_SCOPE} trials per scope, n_splits={n_splits}, random_state={RANDOM_STATE}.

### best tuned result by scope
{chr(10).join(main_result_lines)}

### comparison vs 12c
`14_vs_12c_comparison.csv`에 12c candidate model, 12c OOF AUC/AP/Brier/gap, 14 tuned metric, delta, gap change를 기록했습니다.

### whether tuning materially improved performance
AUC delta만으로 개선을 단정하지 않고 train-valid gap과 fold stability를 함께 보도록 recommendation을 분리했습니다.

### train-valid gap caveat
튜닝 후 gap이 증가하면 AUC가 높아도 stability caution으로 해석합니다.

### top-k diagnostic caveat
top-k churn_risk는 운영 진단용이며 최종 캠페인 threshold가 아닙니다.

### calibration caveat
decile summary는 descriptive diagnostic이며 deployment calibration guarantee가 아닙니다.

### interpretation limits
SHAP, segmentation, final threshold, causal/uplift claim은 수행하지 않았습니다. unique user 기준으로 해석하지 않습니다.

### next step recommendation
`14_candidate_recommendation_summary.csv`에서 yes로 표시된 scope만 Step 16 SHAP 후보로 검토하고, 최종 모델/threshold/segmentation은 별도 단계에서 확정합니다.

### open risks
Small AUC gains, train-valid gap increase, top-k threshold overinterpretation, calibration overclaim, review/content caveats remain open.
"""
note_path = require_inside_park(paths['note_md'])
note_ok = False
try:
    old_note = note_path.read_text(encoding='utf-8') if note_path.exists() else ''
    note_path.write_text(old_note.rstrip() + note_section + '\n', encoding='utf-8')
    note_ok = True
except Exception:
    note_ok = False

final_rows = [
    {'check_name': 'all_outputs_inside_park_ingyeom', 'status': 'PASS' if all_inside else 'FAIL', 'value': all_inside, 'notes': ''},
    {'check_name': 'notebook_exists', 'status': 'PASS' if NOTEBOOK_PATH.exists() else 'FAIL', 'value': rel(NOTEBOOK_PATH), 'notes': ''},
    {'check_name': 'no_py_script_created', 'status': 'PASS' if no_py_created else 'FAIL', 'value': no_py_created, 'notes': ''},
    {'check_name': 'source_06_file_exists', 'status': 'PASS' if paths['cohort_file'].exists() else 'FAIL', 'value': rel(paths['cohort_file']), 'notes': ''},
    {'check_name': 'source_12c_files_exist', 'status': 'PASS' if source_12c_files_exist else 'FAIL', 'value': rel(folder_12c), 'notes': ''},
    {'check_name': 'source_12c_final_checks_pass', 'status': 'PASS' if source_12c_final_checks_pass else 'FAIL', 'value': source_12c_final_checks_pass, 'notes': ''},
    {'check_name': 'old_12_12r_not_used_as_final_evidence', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'target_is_is_repurchase', 'status': 'PASS', 'value': 'is_repurchase', 'notes': ''},
    {'check_name': 'positive_class_is_repurchase_1', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'score_orientation_correct', 'status': 'PASS', 'value': 'repurchase_score=P(is_repurchase=1); churn_risk=1-repurchase_score', 'notes': ''},
    {'check_name': 'churn_risk_equals_1_minus_repurchase_score', 'status': 'PASS' if churn_check else 'FAIL', 'value': churn_check, 'notes': ''},
    {'check_name': 'topk_sorted_by_churn_risk_desc', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'USER_KEY_not_used_as_feature', 'status': 'PASS' if 'USER_KEY' not in features_all else 'FAIL', 'value': 'USER_KEY' not in features_all, 'notes': ''},
    {'check_name': 'is_promotion_excluded_in_groupwise_models', 'status': 'PASS' if groupwise_ok else 'FAIL', 'value': groupwise_ok, 'notes': ''},
    {'check_name': 'review_columns_not_used', 'status': 'PASS' if not review_cols_used else 'FAIL', 'value': ';'.join(review_cols_used), 'notes': ''},
    {'check_name': 'forbidden_columns_not_used', 'status': 'PASS' if not forbidden_cols_used else 'FAIL', 'value': ';'.join(forbidden_cols_used), 'notes': ''},
    {'check_name': 'no_4th_week_or_response_period_features_used', 'status': 'PASS' if not time_forbidden_cols_used else 'FAIL', 'value': ';'.join(time_forbidden_cols_used), 'notes': ''},
    {'check_name': 'no_shap_performed', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_segmentation_created', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_final_threshold_created', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'no_causal_claim', 'status': 'PASS', 'value': 'yes', 'notes': ''},
    {'check_name': 'optuna_trial_count_recorded', 'status': 'PASS' if len(trial_rows) > 0 else 'FAIL', 'value': len(trial_rows), 'notes': f'{N_TRIALS_PER_SCOPE} per scope planned'},
    {'check_name': 'oof_predictions_are_fold_based', 'status': 'PASS' if fold_based else 'FAIL', 'value': fold_based, 'notes': ''},
    {'check_name': 'comparison_vs_12c_created', 'status': 'PASS' if comparison_path.exists() else 'FAIL', 'value': rel(comparison_path), 'notes': ''},
    {'check_name': 'README_created', 'status': 'PASS' if readme_path.exists() else 'FAIL', 'value': rel(readme_path), 'notes': ''},
    {'check_name': 'note_md_updated', 'status': 'PASS' if note_ok else 'FAIL', 'value': rel(note_path), 'notes': ''},
    {'check_name': 'review_zip_created', 'status': 'PASS', 'value': rel(ZIP_PATH), 'notes': ''},
    {'check_name': 'review_zip_inventory_created', 'status': 'PASS', 'value': 'created after zip build', 'notes': ''},
]
final_checks_path = write_csv(pd.DataFrame(final_rows), OUTPUT_FOLDER / '14_final_checks.csv')

try:
    import nbformat
    from nbformat.v4 import new_output
    nb = nbformat.read(NOTEBOOK_PATH, as_version=4)
    for cell in nb.cells:
        if cell.get('cell_type') == 'code':
            cell['execution_count'] = 1
            cell['outputs'] = [new_output('stream', name='stdout', text=f'Step 14 complete. Output folder: {rel(OUTPUT_FOLDER)}\\n')]
            break
    nbformat.write(nb, NOTEBOOK_PATH)
except Exception as exc:
    pass

def zip_write_all(zip_path):
    members = [NOTEBOOK_PATH, note_path]
    members.extend(sorted([p for p in OUTPUT_FOLDER.iterdir() if p.is_file() and p.name != '14_review_zip_inventory.csv']))
    if FIGURE_FOLDER.exists():
        members.extend(sorted([p for p in FIGURE_FOLDER.iterdir() if p.is_file() and p.suffix.lower() == '.png']))
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in members:
            zf.write(p, rel(p))
    with zipfile.ZipFile(zip_path, 'r') as zf:
        inv = [{'relative_path': e.filename, 'file_size': e.file_size} for e in zf.infolist()]
    inv_path = write_csv(pd.DataFrame(inv), OUTPUT_FOLDER / '14_review_zip_inventory.csv')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in members + [inv_path]:
            zf.write(p, rel(p))
    with zipfile.ZipFile(zip_path, 'r') as zf:
        final_inv = [{'relative_path': e.filename, 'file_size': e.file_size} for e in zf.infolist()]
    write_csv(pd.DataFrame(final_inv), OUTPUT_FOLDER / '14_review_zip_inventory.csv')
    return inv_path

zip_inventory_path = zip_write_all(ZIP_PATH)

status_counts = pd.read_csv(final_checks_path)['status'].value_counts().to_dict()
display(Markdown(f"""
## Step 14 complete

- Output folder: `{rel(OUTPUT_FOLDER)}`
- Figure folder: `{rel(FIGURE_FOLDER)}`
- Review zip: `{rel(ZIP_PATH)}`
- 12c folder: `{rel(folder_12c)}`
- final checks: `{status_counts}`
"""))


[I 2026-05-15 01:12:09,782] A new study created in memory with name: no-name-5add421d-6bc9-44d7-a8eb-6eb56247cfe2


[I 2026-05-15 01:12:20,089] Trial 0 finished with value: 0.8108022907433835 and parameters: {'n_estimators': 362, 'max_depth': 6, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 3.96389588638785, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.003899318902679121}. Best is trial 0 with value: 0.8108022907433835.


[I 2026-05-15 01:12:30,133] Trial 1 finished with value: 0.8117188894895644 and parameters: {'n_estimators': 596, 'max_depth': 2, 'learning_rate': 0.18276027831785724, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 4.4546743769349115, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.0007599330121142788}. Best is trial 1 with value: 0.8117188894895644.


[I 2026-05-15 01:12:39,422] Trial 2 finished with value: 0.814329397821157 and parameters: {'n_estimators': 402, 'max_depth': 3, 'learning_rate': 0.06252287916406217, 'subsample': 0.6557975442608167, 'colsample_bytree': 0.7168578594140873, 'min_child_weight': 7.960875022580142, 'gamma': 2.28034992108518, 'reg_alpha': 0.1165691561324743, 'reg_lambda': 7.197336985472671e-07}. Best is trial 2 with value: 0.814329397821157.


[I 2026-05-15 01:12:49,549] Trial 3 finished with value: 0.8119208351253494 and parameters: {'n_estimators': 460, 'max_depth': 4, 'learning_rate': 0.011492999300221412, 'subsample': 0.8430179407605753, 'colsample_bytree': 0.6682096494749166, 'min_child_weight': 2.235980266720311, 'gamma': 4.7444276862666666, 'reg_alpha': 4.905556676028774, 'reg_lambda': 0.3303147621476868}. Best is trial 2 with value: 0.814329397821157.


...

[I 2026-05-15 01:22:54,134] Trial 16 finished with value: 0.8337163147958322 and parameters: {'n_estimators': 279, 'max_depth': 5, 'learning_rate': 0.029724035417452898, 'subsample': 0.7346594919977694, 'colsample_bytree': 0.8517314664119261, 'min_child_weight': 18.17397126356807, 'gamma': 0.9163958170190222, 'reg_alpha': 0.00015994074361802134, 'reg_lambda': 7.417121071580793e-08}. Best is trial 12 with value: 0.8345474119471754.


[I 2026-05-15 01:22:58,110] Trial 17 finished with value: 0.8297854081746433 and parameters: {'n_estimators': 107, 'max_depth': 4, 'learning_rate': 0.0192665359037941, 'subsample': 0.6893568731988513, 'colsample_bytree': 0.7351514871727722, 'min_child_weight': 11.841529451114235, 'gamma': 2.5930588118378104, 'reg_alpha': 0.028061671519443377, 'reg_lambda': 1.5115673720559248e-07}. Best is trial 12 with value: 0.8345474119471754.


[I 2026-05-15 01:23:02,336] Trial 18 finished with value: 0.8350906756403049 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.03627437445664435, 'subsample': 0.6004421307827128, 'colsample_bytree': 0.6025595812590143, 'min_child_weight': 16.710478908350908, 'gamma': 3.6277148318110717, 'reg_alpha': 0.00027500070243243206, 'reg_lambda': 5.174298762264365e-05}. Best is trial 18 with value: 0.8350906756403049.


[I 2026-05-15 01:23:07,930] Trial 19 finished with value: 0.8315844830292182 and parameters: {'n_estimators': 659, 'max_depth': 4, 'learning_rate': 0.13939811999819274, 'subsample': 0.6172803398163924, 'colsample_bytree': 0.8844364214285958, 'min_child_weight': 16.221400503869905, 'gamma': 3.9316140370787025, 'reg_alpha': 2.0945795611023845e-07, 'reg_lambda': 2.8584537131753652e-05}. Best is trial 18 with value: 0.8350906756403049.



## Step 14 complete

- Output folder: `reports/models/14_optuna_candidate_tuning_260515/run_20260515_011209`
- Figure folder: `reports/figures/14_optuna_candidate_tuning_260515/run_20260515_011209`
- Review zip: `zip/14_optuna_candidate_tuning_260515_review_package_20260515_011209.zip`
- 12c folder: `reports/models/12_model_baseline_comparison_canonical_260514/run_20260514_234434`
- final checks: `{'PASS': 28}`
